<a href="https://colab.research.google.com/github/marcohuertas/AI-agents-projects/blob/main/agentic_RAG_systems_biomodels_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AGENTIC RAG FOR MODEL RETRIEVAL

This notebook is part of an exercise in creating an Agenic Rag, that uses the user query about a model description and provides a suggestion on the best match based on a pre-designed HF dataset.


Code follows the setup in https://huggingface.co/learn/agents-course/unit2/smolagents/retrieval_agents

## INSTALL PACKAGES

In [ ]:
! pip install --quiet --upgrade langchain-community
! pip install -qU langchain-huggingface
! pip install --quiet --upgrade 'smolagents[transformers]' accelerate bitsandbytes
! pip install --quiet faiss-cpu
# ! pip install --quiet faiss-gpu

# IMPORT DATASET

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from datasets import load_dataset

ds = load_dataset("m1969m/biomodels-sbml-odemodels-oncology", split='train')
model_descriptions = ds.select_columns(['id', 'name', 'synopsis', 'abstract', 'antimony_string', 'pmcid', 'accession'])
model_descriptions = model_descriptions.map(lambda u: {'title': u['name'].split(" - ")[-1]})

# Code improved by Claude (Anthropic), based on prior version
def extract_reactions(antimony_string: str) -> str:
    """Extract the Reactions section from an Antimony string."""
    if not antimony_string:
        return "No reactions available."

    chunks = antimony_string.split('//')
    for chunk in chunks:
        if 'Reactions:' in chunk:
            return chunk.strip()

    return "No reactions found in Antimony string."


def concatenate_text(sample):
    description = sample['abstract'] if sample['abstract'] != 'NoAbstract' else sample['synopsis']
    reactions = extract_reactions(sample['antimony_string'])

    text = (
        f"{sample['title']}\n\n"
        f"=== Model Description ===\n{description}\n\n"
        f"=== Model Reactions ===\n{reactions}"
    )
    return {'text': text, 'description': description, 'reactions': reactions}

model_descriptions = model_descriptions.map(concatenate_text)

# PREPARE A VALIDATION DATASET
For each of 5 categories, prepare queries that correspond to specific model ids. The agent should extract the correct model id based on the semanitc similarity of the query with the model description.

Avoid accronyms and write them from a biologist point of view looking for models to use.

In [ ]:
import pandas as pd
import textwrap

In [ ]:
df = model_descriptions.select_columns(['id', 'name', 'description', 'reactions']).to_pandas()
df = df[df.reactions!='No reactions available.']
df.drop_duplicates(subset='description', inplace=True)

In [ ]:
from collections import defaultdict

def prepare_validation_df(dfref, descr_list):
  dref = defaultdict(list)
  for s, descr in zip(dfref.iterrows(), descr_list):
    dref['id'].append(s[1]["id"])
    dref['descr_query'].append(descr)

  return pd.DataFrame.from_dict(dref)

## Category : CAR (chimeric antigen receptor)

In [ ]:
dfcar = df[df["description"].apply(lambda u: "CAR" in u)]
idlist = dfcar.id.values
descrlist = dfcar.description.values

In [ ]:
posqueries_dict = {
  "idlist": [
    ["BIOMD0000001041"],
    ["BIOMD0000001013"],
    ["BIOMD0000000837"],
  ],
  "query": [
    """I'm searching for a model that describes the interplay between
    endogenous T cells and chimeric antigen receptor T cells and their interaction
    with cancer cells and study dosing approaches that can provide the maximum benefit.""",
    """I'm looking for an example model that describes how the tumor
    microenvironment influences the effectiveness of chimeric antigen receptor T cells
    in the contexte of non-hematological cancers and explores conditions under which this
    injected T cells could become more effective in killing tumor cells.""",
    """I'm interested in modeling the interaction between chimeric
    antigen receptor T cells in blood cancers cells with an emphasis on the hazardous
    inflamatory effects of cytokines in this therapies. """
  ],
  "query_type": ["pos"]*3,
  "category": ["car"]*3,

}

negqueries_dict = {
  "idlist":[['BIOMD0000001041', 'BIOMD0000001013', 'BIOMD0000001011',
       'BIOMD0000001024', 'BIOMD0000000837']],
  "query": [
      """I'm searching for a model that describes the interaction of
      immune cells and blood cancer cells under various therapies excluding those
      using chimeric antigen receptor T cells.
      """],
    "query_type": ["neg"],
    "category": ["car"],
  }

dfcar_val = pd.concat([
    pd.DataFrame.from_dict(posqueries_dict),
    pd.DataFrame.from_dict(negqueries_dict),
])

dfcar_val

## Category : Interleukin

In [ ]:
dfil = df[df["description"].apply(lambda u: "interleukin" in u.lower())]
idlist = dfil.id.values
descrlist = dfil.description.values

In [ ]:
posqueries_dict = {
  "idlist": [
    ["BIOMD0000000881"],
    ["BIOMD0000000913"],
    ["BIOMD0000000910"],
    ["BIOMD0000000761"]
  ],
  "query": [
    """I'm looking for a model that explore the interactions between
    the two types of helper T cells and with malignant cancer cells and how the proportion
    of each and their effectiveness against melanoma is influenced by the cytokine interleukin.""",
    """I'm searching for a model that describes interactions between tumor and immune cells,
    that I can use to explore the effects between various therapies,
    like chemo and immune therapies and perhaps also cytokine therapies, paying attention
    to the role that interleukin may play.""",
    """I need a model that takes into consideration various expression levels of
    antigens in tumor cells and the effect cytokines, like interleuking, might play
    in the immune response against tumors.""",
    """I'm interested in a model that explores the influence of various
    forms of interleukin in the tumor suppression driven by natural killer cells
    and endogenous T cells when cytokine therapies are used.""",
  ],
  "query_type": ["pos"]*4,
  "category": ["interleukin"]*4,

}

negqueries_dict = {
  "idlist":[
      ["BIOMD0000000910"],
      ],
  "query": [
      """I'm looking for models that take into consideration the dynamics and effects
      of interleukin in the interaction of the immune system and tumor cells and that
      explore various therapies that do not involve the use of electromagnetic waves.""",
      ],
    "query_type": ["neg"],
    "category": ["interleukin"],

  }

dfil_val = pd.concat([
    pd.DataFrame.from_dict(posqueries_dict),
    pd.DataFrame.from_dict(negqueries_dict),
])

dfil_val

## Category : Breast Cancer

In [ ]:
dfbreast = df[df["description"].apply(lambda u: "breast" in u.lower())]
idlist = dfbreast.id.values
descrlist = dfbreast.description.values
print(len(descrlist))

In [ ]:
posqueries_dict = {
  "idlist": [
    ["BIOMD0000001033"],
    ["MODEL1912120005"],
    ["MODEL1909090002"]
  ],
  "query": [
    """I need a model that focuses on describing how various types of macrophages
    in the tumor microenvironment interact with breast cancer cells and how they might
    influence the effectivess of cancer tharpies that rely on the use of viruses to
    kill the cancer cells.""",
    """I searching for models developed to describe various forms of therapies
    used in the fight against breast cancer that do not involve immuno or oncolytic
    therapies.""",
    """I'm interested in finding a mathematical model that describes the interactions
    between various immune cells and cancer cells, particularly in typical
    breast cancer cell lines, and that includes the effect of relevant hormones."""
  ],
  "query_type": ["pos"]*3,
  "category": ["breast cancer"]*3,
}

negqueries_dict = {
  "idlist":[
      ['BIOMD0000001033', 'MODEL1912120005', 'MODEL1909090002',
       'BIOMD0000000745'],
      ["MODEL1912120005"],
      ],
  "query": [
      """I'm looking for mathematical models that interactions between immune cells,
      like natural killer cells, and cancer cells with applications in various cancers,
      except breast cancer.""",
      """I searching for models developed to describe various forms of therapies
      used against breast cancer cells that involve some type of immune cells, like macrophages or
      natural killer cells or oncolytic viruses."""
      ],
    "query_type": ["neg"]*2,
    "category": ["breast cancer"]*2,
  }

dfbreast_val = pd.concat([
    pd.DataFrame.from_dict(posqueries_dict),
    pd.DataFrame.from_dict(negqueries_dict),
])

dfbreast_val

## Category : Pancreatic Cancer

In [ ]:
dfpanc = df[df["description"].apply(lambda u: "pancreatic" in u.lower())]
idlist = dfpanc.id.values
descrlist = dfpanc.description.values
print(len(descrlist))

In [ ]:
posqueries_dict = {
  "idlist": [
    ["BIOMD0000000744"],
    ["BIOMD0000000811"],
    ["MODEL1909100002"],
  ],
  "query": [
    """I need a model of pancreatic cancer that captures the role of the
    tumor microenvironment, specifically the interplay between pro- and
    anti-tumor cytokines and immune effector cells, to evaluate adoptive cell
    transfer strategies""",
    """Looking for a mathematical model that accounts for immune suppression
    by regulatory T cells in pancreatic cancer, and can be used to compare the
    effect of single vs combined immunotherapy schedules on overall survival""",
    """I need a mathematical model that analyses immune response evation seen in
    pancreatic cancers and the role played by macrophages and T lymphocytes in
    killing tumor cells.""",
  ],
  "query_type": ["pos"]*3,
  "category": ["pancreatic cancer"]*3,

}

negqueries_dict = {
  "idlist":[
      ["BIOMD0000000744",
      "BIOMD0000000811",
      "MODEL1909100002"],
      ],
  "query": [
      """I'm looking for a pancreatic cancer model focused on drug PK/PD and
       tumor growth inhibition, without an immune system component"""
      ],
    "query_type": ["neg"],
    "category": ["pancreatic cancer"],

  }

dfpanc_val = pd.concat([
    pd.DataFrame.from_dict(posqueries_dict),
    pd.DataFrame.from_dict(negqueries_dict),
])

dfpanc_val

## Category :  Leukemia

In [ ]:
dfleuk = df[df["description"].apply(lambda u: "leukemia" in u.lower())]
idlist = dfleuk.id.values
descrlist = dfleuk.description.values
print(len(descrlist))

In [ ]:
posqueries_dict = {
  "idlist": [
    ["BIOMD0000000879"],
    ["MODEL2001090002"]
  ],
  "query": [
    """I'm interested in a model of chronic lymphocytic leukemia that focuses
    on chemotherapy and immunotherapy combinations.""",
    """I'm interested a disease progression model of chronic lymphocytic leukemia
    that focuses on describing the actions of immune cells only.""",
  ],
  "query_type": ["pos"]*2,
  "category": ["leukemia"]*2,

}

negqueries_dict = {
  "idlist":[
      ["BIOMD0000000879"],
      ],
  "query": [
    """I'm interested a disease progression model of chronic lymphocytic leukemia
    that focuses on describing the actions of immune cells only."""
      ],
    "query_type": ["neg"],
    "category": ["leukemia"],

  }

dfleuk_val = pd.concat([
    pd.DataFrame.from_dict(posqueries_dict),
    pd.DataFrame.from_dict(negqueries_dict),
])

dfleuk_val

Concatenating validation set

In [ ]:
dfval_queries = pd.concat(
  [
    dfcar_val,
    dfil_val,
    dfbreast_val,
    dfpanc_val,
    dfleuk_val,
  ]
).reset_index(drop=True)

dfval_queries

Sample one of the positive queries from each category


In [ ]:
dfval_queries_pos = (
  dfval_queries[dfval_queries.query_type=="pos"]
 .groupby('category')
 .apply(lambda df: df.sample(n=1, replace=False, random_state=1969), include_groups=False)
 .reset_index().drop('level_1', axis=1)
 )[dfval_queries.columns]

dfval_queries_pos

In [ ]:
dfval_queries_neg = (
  dfval_queries[dfval_queries.query_type=="neg"]
 .groupby('category')
 .apply(lambda df: df.sample(n=1, replace=False, random_state=1969), include_groups=False)
 .reset_index().drop('level_1', axis=1)
 )[dfval_queries.columns]

dfval_queries_neg

# PREPARE A VECTORSTORE

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

from smolagents import Tool, ToolCallingAgent, tool, InferenceClientModel, LiteLLMModel

import textwrap

Prepare documents from HF dataset that can be used for retrieval

In [ ]:
# Build documents with rich metadata
source_docs = []
for idx in range(model_descriptions.shape[0]):
  row = model_descriptions[idx]
  source_docs.append(
      Document(
          page_content=row["text"],
          metadata={
              "source": row["name"],
              "title": row["title"],
              "accession": row["accession"],
              "pmcid": row["pmcid"],
              "biomodelid": row["id"],
          }
      )
  )

# Build FAISS index — runs on CPU, no GPU needed

model_name_1 = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
model_name_2 = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model_1 = HuggingFaceEmbeddings(
  model_name=model_name_1,
  model_kwargs={"device": "cpu"}
)

embedding_model_2 = HuggingFaceEmbeddings(
  model_name=model_name_2,
  model_kwargs={"device": "cpu"}
)

vectorstore_1 = FAISS.from_documents(source_docs, embedding_model_1)
# vectorstore_2 = FAISS.from_documents(source_docs, embedding_model_2)

# AGENT

Prepare tool for agent

In [ ]:
# Code structure suggested by Claude (Anthropic)
class BioModelsRetrieverTool(Tool):
    name = "biomodels_retriever"
    description = """
    Searches a curated database of ODE-based oncology models from BioModels.
    Returns the most relevant model(s) including their description, reactions,
    accession ID, and source publication. Use this tool to find mechanistic
    models matching a biological question.
    """
    inputs = {
        "query": {
            "type": "string",
            "description": "Natural language description of the biological mechanism or model you are looking for.",
        }
    }
    output_type = "string"

    def __init__(self, vectorstore, n_results=3, **kwargs):
        super().__init__(**kwargs)
        self.vectorstore = vectorstore
        self.n_results = n_results

    def forward(self, query: str) -> str:
        assert isinstance(query, str), "Query must be a string."

        docs = self.vectorstore.similarity_search(query, k=self.n_results)

        if not docs:
            return "No relevant models found for the given query."

        output_parts = []
        for i, doc in enumerate(docs, 1):
            meta = doc.metadata
            output_parts.append(
                f"--- Result {i} ---\n"
                f"Accession: {meta.get('accession', 'N/A')}\n"
                f"Title: {meta.get('title', 'N/A')}\n"
                f"BioModelId: {meta.get('biomodelid', 'N/A')}\n"
                f"PMCID: {meta.get('pmcid', 'N/A')}\n\n"
                f"{doc.page_content}\n"
            )

        return "\n".join(output_parts)


### Load Model
Either using TransformersModel or InferenceClientModel

### Using TransformersModel
Loading model directly, usually not enough space on Google Colab

In [ ]:
# Uncomment if running locally on GPU of adequate size

# from smolagents import TransformersModel
# import torch
# from transformers import BitsAndBytesConfig
# from transformers import AutoTokenizer

# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     # bnb_4bit_compute_dtype=torch.float16,
#     # bnb_4bit_quant_type="nf4",
#     # bnb_4bit_use_double_quant=True,
# )

# model_id = "Qwen/Qwen2.5-7B-Instruct"

# model = TransformersModel(
#     model_id=model_id,
#     device_map='auto',
#     model_kwargs={"quantization_config": quantization_config},
#     # device='cuda'
# )

### Inference Model

In [ ]:
# model_id = "meta-llama/Llama-3.3-70B-Instruct"
model_id = "Qwen/Qwen2.5-72B-Instruct"

model = InferenceClientModel(
    model_id=model_id,
    # token=your_hf_token
)


In [ ]:
# Instantiate
retriever_tool = BioModelsRetrieverTool(vectorstore_1, n_results=5)

agent = ToolCallingAgent(
    tools=[retriever_tool],
    model=model,
    max_steps=5  # prevent runaway loops
)

### PREPARE VALIDATION QUERY

In [ ]:
dfval_queries.category.unique().tolist()

### Positive query

In [ ]:
category = 'leukemia'
dfpos = dfval_queries_pos[dfval_queries_pos.category==category] # randomly selected in earlier cell
# dfpos = dfval_queries[(dfval_queries.category==category) & (dfval_queries.query_type=='pos')] # all queries
print(dfpos.shape)

idx = 0
query = dfpos["query"].values[idx]
modelid = dfpos["idlist"].values[idx]

print(modelid)

pos_query_full = f"""
{query}
Use the tools provided to you to find a suitable model.
For your final answer, explain why this model is the best option, include
the source publication (PMCID), the biomodel id (BioModelId) and the
reactions of the model. If no reactions are available, indicate that explicitly.
"""

print(pos_query_full)

In [ ]:
response = agent.run(pos_query_full)

In [ ]:
print(textwrap.fill(response,100))

### Negative query

In [ ]:
# dfpos = dfval_queries_neg[dfval_queries_neg.category==category] # using a randomly selected one in a previous cell
dfpos = dfval_queries[(dfval_queries.category==category) & (dfval_queries.query_type=='neg')] # all queries
print(dfpos.shape)

idx = 0
query = dfpos["query"].values[idx]
modelid = dfpos["idlist"].values[idx]

print(category)
print(modelid)

neg_query_full = f"""
{query}
Use the tools provided to you to find a suitable model.
For your final answer, explain why this model is the best option, include
the source publication (PMCID), the biomodel id (BioModelId) and the
reactions of the model. If no reactions are available, indicate that explicitly.
"""

print(neg_query_full)

In [ ]:
response_neg = agent.run(neg_query_full)

In [ ]:
print(textwrap.fill(str(response_neg),100))